In [2]:
from qgee.qubit_hamil import MolecularQubitHamiltonian, geometry_for_molecule

from pyscf import gto, scf, fci
import numpy as np

import pandas as pd

In [3]:
def get_fci_energy(name, basis="sto3g"):
    geometry = geometry_for_molecule(name)

    mol = gto.M(
        atom=geometry,
        basis=basis,
        charge=0,
        spin=0,
        unit="Angstrom",
        verbose=0,
    )

    mf = scf.RHF(mol).run()

    solver = fci.FCI(mf)
    energy, ci = solver.kernel()

    return {
        "molecule": name,
        "FCI total energy (Eh)": energy,
        "nuclear repulsion (Eh)": mol.energy_nuc(),
        "electronic energy (Eh)": energy - mol.energy_nuc(),
    }

results = [
    get_fci_energy(name)
    for name in ["H2", "LiH", "BeH2"]
]

pd.DataFrame(results)

,molecule,FCI total energy (Eh),nuclear repulsion (Eh),electronic energy (Eh)
0,H2,-1.137274,0.714139,-1.851414
1,LiH,-7.882402,0.995318,-8.877720
2,BeH2,-15.595182,3.392162,-18.987344


In [4]:
def compare_energies(name, basis="sto3g"):
    geometry = geometry_for_molecule(name)

    hamiltonian = MolecularQubitHamiltonian(
        geometry,
        basis=basis,
        charge=0,
        spin=0,
    )

    fci_result = get_fci_energy(name, basis)

    e_qubit_electronic = hamiltonian.exact_ground_state_energy()
    e_qubit_total = (
        e_qubit_electronic
        + fci_result["nuclear repulsion (Eh)"]
    )

    return {
        "molecule": name,
        "qubits": hamiltonian.num_qubits,
        "FCI total (Eh)": fci_result["FCI total energy (Eh)"],
        "Qubit ED total (Eh)": e_qubit_total,
        "Difference (mEh)": (
            e_qubit_total - fci_result["FCI total energy (Eh)"]
        ) * 1000,
    }

comparison = pd.DataFrame([
    compare_energies(name)
    for name in ["H2", "LiH", "BeH2"]
])

comparison

,molecule,qubits,FCI total (Eh),Qubit ED total (Eh),Difference (mEh)
0,H2,4,-1.137274,-1.137274,3.774758e-12
1,LiH,12,-7.882402,-7.882402,-4.085621e-11
2,BeH2,14,-15.595182,-15.595182,-1.882938e-10


The qubit hamiltonian and FCI ground state calculation agree very closely -- very good indicator for our Hamiltonian construction. 

This can be compared with our QSE and QPE results.

In [8]:
def get_elec(name, basis="sto3g"):
    geometry = geometry_for_molecule(name)

    hamiltonian = MolecularQubitHamiltonian(
        geometry,
        basis=basis,
        charge=0,
        spin=0,
    )

    return {
        "molecule": name,
        "electronic energy": hamiltonian.exact_ground_state_energy()
    }


elec_energies = pd.DataFrame([
    get_elec(name)
    for name in ["H2", "LiH", "BeH2"]
])

elec_energies

,molecule,electronic energy
0,H2,-1.851414
1,LiH,-8.877720
2,BeH2,-18.987344
